In [7]:
import pandas as pd
df = pd.read_sas("../data/raw/P_OHXDEN.XPT")
ctc_cols = [c for c in df.columns if c.endswith('CTC')]
has_untreated_coronal = (
    df[ctc_cols]
    .apply(lambda row: (row == b'U').any(), axis=1)
)
has_root_caries = df["OHXRCAR"] == 1

df["has_caries"] = (has_root_caries | has_untreated_coronal).astype(int)

In [8]:
tc_cols = [c for c in df.columns if c.endswith("TC") and not c.endswith("CTC")]

df["n_missing_teeth"] = (df[tc_cols] == 4).sum(axis=1)

In [9]:
df["n_filled_teeth"] = (df[ctc_cols] == b'F').sum(axis=1)
df["n_untreated_teeth"] = (df[ctc_cols] == b'U').sum(axis=1)
df["has_root_caries"] = (df["OHXRCAR"] == 1).astype(int)
df[
    ["n_missing_teeth", "n_filled_teeth", "n_untreated_teeth", "has_root_caries"]
].describe()
df.head()

,SEQN,OHDEXSTS,OHDDESTS,OHXIMP,OHX01TC,OHX02TC,OHX03TC,OHX04TC,OHX05TC,OHX06TC,...,OHX21SE,OHX28SE,OHX29SE,OHX30SE,OHX31SE,has_caries,n_missing_teeth,n_filled_teeth,n_untreated_teeth,has_root_caries
0,109263.0,1.0,1.0,NaN,4.0,4.0,4.0,1.0,1.0,1.0,...,b'',b'',b'',b'',b'',1,12,0,8,0
1,109264.0,1.0,1.0,2.0,4.0,2.0,2.0,2.0,2.0,2.0,...,b'0',b'0',b'0',b'0',b'0',0,4,0,0,0
2,109265.0,1.0,1.0,NaN,4.0,4.0,4.0,1.0,1.0,1.0,...,b'',b'',b'',b'',b'',1,14,0,10,0
3,109266.0,1.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,b'',b'',b'',b'',b'',0,0,8,0,0
4,109269.0,1.0,1.0,NaN,4.0,4.0,4.0,1.0,1.0,1.0,...,b'',b'',b'',b'',b'',1,12,0,8,0


In [ ]:
import pyarrow
features = [
    "SEQN",
    "has_caries",
    "n_missing_teeth",
    "n_filled_teeth",
    "n_untreated_teeth",
    "has_root_caries",
]

df_model = df[features]

df_model.to_parquet("../data/processed/caries_features.parquet", index=False)
